### Chapter 6.6 - File I/O

File I/O means saving data to disk and loading it later. PyTorch can save tensors and model parameters for reuse.


In [ ]:
from pathlib import Path
import tempfile
import torch


### 6.6.1 Loading and Saving Tensors

#### 1. Intuition

Saving a tensor writes its values to a file. Loading reads those values back into memory.

A file path tells the program where the saved object lives.

#### 2. Why this exists

Training can take time, and intermediate data may need to be reused. Saving avoids recomputing or losing results.


#### 3. Examples

Save and load one tensor in a temporary directory.


In [ ]:
with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "x.pt"
    x = torch.tensor([1.0, 2.0])
    torch.save(x, path)
    loaded = torch.load(path)
loaded


Save and load a dictionary of tensors.


In [ ]:
with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "data.pt"
    torch.save({"a": torch.ones(2)}, path)
    loaded = torch.load(path)
loaded["a"]


#### 4. Step-by-step breakdown

`TemporaryDirectory` creates a short-lived folder.

`torch.save(x, path)` writes the tensor.

`torch.load(path)` reads it back.

A dictionary can store multiple named tensors in one file.

#### 5. Connection to ML systems

Tensor saving is useful for cached data, embeddings, checkpoints, and debugging artifacts.

#### 6. Common confusion points

- Save to known paths in real projects.
- Temporary directories disappear after the block.
- `torch.load` reads Python objects saved by PyTorch.
- Be careful loading files from untrusted sources.


### 6.6.2 Loading and Saving Model Parameters

#### 1. Intuition

A model's `state_dict` is a dictionary containing parameter and buffer tensors.

A buffer is persistent tensor state that is not usually trained by gradients.

#### 2. Why this exists

Saving only parameters lets you recreate the model architecture in code and restore learned values from disk.


#### 3. Examples

Save and load a model state dictionary.


In [ ]:
net = torch.nn.Linear(2, 1)
with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "model.pt"
    torch.save(net.state_dict(), path)
    new_net = torch.nn.Linear(2, 1)
    new_net.load_state_dict(torch.load(path))
new_net.weight.shape


#### 4. Step-by-step breakdown

`state_dict()` returns the model's saved tensor state.

A new model with the same architecture is created.

`load_state_dict` copies saved values into the new model.

The architecture must match the saved parameter shapes.

#### 5. Connection to ML systems

Model checkpoints usually store `state_dict`s, optimizer state, epoch number, and metadata.

#### 6. Common confusion points

- Saving parameters is not the same as saving the class definition.
- The loading model must have compatible architecture.
- Optimizer state is separate from model state.
- Checkpoints should include enough metadata to resume work.


### 6.6.3 Summary

#### 1. Intuition

PyTorch file I/O commonly uses `torch.save`, `torch.load`, `state_dict`, and `load_state_dict`.

#### 2. Why this exists

Saving and loading make experiments reproducible and training resumable.


#### 3. Examples

A checkpoint dictionary structure.


In [ ]:
checkpoint = {
    "model": "state_dict goes here",
    "epoch": 3,
    "note": "tiny example",
}
checkpoint


#### 4. Step-by-step breakdown

The checkpoint dictionary names what would be saved.

Real code would store actual tensor states.

Metadata such as epoch helps resume training.

#### 5. Connection to ML systems

Well-designed checkpoint files are part of reliable ML engineering.

#### 6. Common confusion points

- Save enough information to reproduce results.
- Keep architecture code versioned.
- Do not trust arbitrary serialized files.
- Verify loaded models with a tiny prediction.


### 6.6.4 Exercises

#### 1. Intuition

These exercises practice local saving and loading.

#### 2. Why this exists

File I/O should be tested with tiny artifacts before it is trusted for large checkpoints.


#### 3. Examples

Exercise 1: save and load a scalar tensor.


In [ ]:
with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "scalar.pt"
    torch.save(torch.tensor(7.0), path)
    value = torch.load(path)
value


Exercise 2: inspect state dictionary keys.


In [ ]:
net = torch.nn.Linear(3, 2)
list(net.state_dict().keys())


#### 4. Step-by-step breakdown

Exercise 1 checks tensor serialization.

Exercise 2 checks model state naming.

Linear layers usually expose `weight` and `bias` in their state dictionary.

#### 5. Connection to ML systems

These checks prepare for saving larger models and checkpoints.

#### 6. Common confusion points

- Temporary files are only for examples.
- State dictionary keys depend on module names.
- Loading requires compatible shapes.
- Keep save paths organized in real projects.
